# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore the schema with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s using the Croissant schema.

*Note*: All references are by `@id`.

Let's enumerate all record sets in the dataset and inspect their fields and columns. 

In [ ]:
# List all record sets by `@id`
record_sets = dataset.metadata.recordSet
print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', rs['@id'])}")

# For each record set, list their fields and columns
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    print("Fields:")
    for field in fields:
        print(f"  - @id: {field['@id']}")
    columns = rs.get('column', [])
    print("Columns:")
    for col in columns:
        print(f"  - @id: {col['@id']} | name: {col.get('name', col['@id'])}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field/column `@id`s from the data overview above.

In [ ]:
# Extract each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, and grouping.

*All references use `@id` for record sets and columns.*

In [ ]:
# Example: Select a numeric field for filtering/normalization
# Let's pick a record set and numeric field for demonstration:
# (Please check the printed columns above for the right @id keys)

# For illustration, assume one record set has a numeric column @id 'age_at_diagnosis'
selected_record_set_id = record_set_ids[0]  # Change as needed for your analysis
df = dataframes[selected_record_set_id]

# Find numeric columns
numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or df[col].apply(lambda x: str(x).replace('.', '', 1).isdigit()).all()]
print(f"Numeric columns in RecordSet {selected_record_set_id}: {numeric_cols}")
if numeric_cols:
    numeric_field_id = numeric_cols[0]
    # Convert column to numeric if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by another field if present (e.g. 'infertility')
    group_field = 'infertility'  # Replace with relevant @id if necessary
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or relationships between fields with `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric_field distribution
if numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot if more than one numeric field
    if len(numeric_cols) > 1:
        second_numeric = numeric_cols[1]
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[numeric_field_id], y=df[second_numeric])
        plt.title(f"{numeric_field_id} vs {second_numeric} in RecordSet {selected_record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(second_numeric)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, reviewing, and exploring the FAIR^2 dataset using the `mlcroissant` library. All entities such as record sets, fields, and columns were referenced by their `@id`. You can extend the analyses further by exploring additional record sets, categorical fields, and customizing filters or visualizations as needed based on the data schema.